In [1]:
from pymongo import MongoClient

# Usamos el nombre del contenedor que Docker reconoce internamente
client = MongoClient('mongodb://bigdata_mongodb:27017/')

try:
    db = client['Canasta_db']
    coleccion = db['Retail_A']
   
    # Prueba de fuego
    client.admin.command('ping')
    print("✅ ¡CONECTADO! Ahora los datos sí llegarán a Mongo Express.")
   
    # Aquí va tu lógica de insert_many(datos_finales)
    if datos_finales:
        coleccion.insert_many(datos_finales)
        print(f"📦 {len(datos_finales)} productos guardados.")
       
except Exception as e:
    print(f"❌ Error de conexión: {e}")

❌ Error de conexión: bigdata_mongodb:27017: [Errno -2] Name or service not known (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69e926b49ead840b648596b7, topology_type: Unknown, servers: [<ServerDescription ('bigdata_mongodb', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('bigdata_mongodb:27017: [Errno -2] Name or service not known (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>


In [2]:
# --- SCRAPING TOTTUS CARNES ---
import os
import time
import random
import re
import json
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bson import ObjectId

# --- LIMPIEZA DE ENTORNO ---
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("✅ Entorno limpio.")

# --- VARIABLES ---
NOMBRE_GRUPO = "Vannessa"
CATEGORIA = "Carnes"
URL_TOTTUS = "https://www.tottus.cl/tottus-cl/lista/CATG27069/Carnes"
datos_finales = []

# --- CONFIGURACIÓN ---
options = Options()
# Si estás en Windows, comenta la línea de binary_location
# options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36")

def serializar_datos(lista):
    """Convierte ObjectIds de Mongo a strings para que JSON pueda guardarlos."""
    for item in lista:
        if "_id" in item:
            item["_id"] = str(item["_id"])
    return lista

driver = webdriver.Chrome(options=options)

try:
    print(f"🚀 Cargando {CATEGORIA} en Tottus...")
    driver.get(URL_TOTTUS)
   
    # Espera explícita de hasta 20 segundos a que aparezca la grilla principal
    wait = WebDriverWait(driver, 20)

    # 1. Scroll de carga con pausas más largas
    print("🖱️ Bajando para forzar el renderizado...")
    for i in range(4):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) # Pausa necesaria para que el servidor responda con datos

   # 2. SCROLL PASO A PASO (Crucial para Tottus)
    print("🖱️ Bajando por la página para activar la grilla...")
    for i in range(8):  # Aumentamos a 8 scrolls
        # Bajamos de a 600 pixeles para que a la página le de tiempo de cargar
        driver.execute_script(f"window.scrollBy(0, 700);")
        print(f"   Bajando... {i+1}/8")
        time.sleep(2) # Espera para que el "Cargando..." desaparezca

# 3. EXTRACCIÓN MAESTRA (Enfoque de Búsqueda de Texto Profundo)
    print("🔍 Iniciando escaneo profundo de texto...")
   
    script_total = """
    let listado = [];
    // Buscamos TODOS los elementos que contengan un signo peso
    let todos = document.querySelectorAll('span, div, b, p, strong');
   
    todos.forEach(el => {
        if (el.innerText.includes('$') && el.innerText.length < 20) {
            // Cuando encontramos un precio, subimos al contenedor padre para agarrar el producto completo
            let contenedor = el.parentElement;
            for (let i = 0; i < 5; i++) { // Subimos hasta 5 niveles para encontrar el cuadro del producto
                if (contenedor && (contenedor.innerText.length > 50)) {
                    listado.push(contenedor.innerText);
                    break;
                }
                contenedor = contenedor.parentElement;
            }
        }
    });
    return [...new Set(listado)];
    """
   
    bloques_texto = driver.execute_script(script_total)
    print(f"📦 Bloques detectados en escaneo profundo: {len(bloques_texto)}")

    for i, texto_raw in enumerate(bloques_texto):
        try:
            lineas = [l.strip() for l in texto_raw.split('\n') if len(l.strip()) > 1]
           
            # Filtro: debe tener al menos un precio
            precio = next((l for l in lineas if '$' in l), None)
           
            # El nombre suele ser la primera o segunda línea que no es basura
            palabras_prohibidas = ["DESPACHO", "RETIRO", "AGREGAR", "VER MÁS", "ORDENAR", "FILTRAR", "CLIENTE"]
            candidatos = [l for l in lineas if '$' not in l and len(l) > 8
                          and not any(p in l.upper() for p in palabras_prohibidas)]
           
            if precio and candidatos:
                nombre_final = candidatos[0]
                # Si es una marca corta (ej: "ARIZTIA"), sumamos la siguiente línea
                if len(nombre_final) < 12 and len(candidatos) > 1:
                    nombre_final = f"{nombre_final} {candidatos[1]}"

                valor_limpio = re.sub(r'[^\d]', '', precio)
               
                datos_finales.append({
                    "identificador": nombre_final,
                    "valor": precio,
                    "valor_limpio": float(valor_limpio),
                    "categoria": CATEGORIA,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO,
                    "tienda": "Tottus",
                    "url": URL_TOTTUS
                })
                print(f"🥩 [{len(datos_finales)}] {nombre_final[:40]} | {precio}")
        except:
            continue
    # 4. Extracción con validación de contenido
    for i, prod in enumerate(productos):
        try:
            texto_raw = prod.text
            if not texto_raw or '$' not in texto_raw:
                continue
               
            lineas = [l.strip() for l in texto_raw.split('\n') if l.strip()]
           
            # Buscamos el precio (línea con $)
            precio = next((l for l in lineas if '$' in l), None)
           
            # Buscamos el nombre (línea con más de 10 caracteres, sin $, sin palabras de sistema)
            palabras_ruido = ["ORDENAR", "FILTRAR", "AGREGAR", "VER MÁS", "DESPACHO", "RETIRO"]
            nombres_posibles = [l for l in lineas if '$' not in l and len(l) > 10
                                and not any(x in l.upper() for x in palabras_ruido)]
           
            nombre = nombres_posibles[0] if nombres_posibles else "Producto no identificado"

            if precio and nombre != "Producto no identificado":
                valor_limpio = re.sub(r'[^\d]', '', precio)
               
                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio,
                    "valor_limpio": float(valor_limpio) if valor_limpio else 0.0,
                    "categoria": CATEGORIA,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO,
                    "tienda": "Tottus",
                    "url": URL_TOTTUS,
                    "posicion": len(datos_finales) + 1
                })
                print(f"🥩 [{len(datos_finales)}] {nombre[:40]} | {precio}")
        except:
            continue

finally:
    driver.quit()
    print("\nNavegador cerrado.")

# --- PASO 5: GUARDADO Y BACKUP ---
if datos_finales:
    # Intento de guardado en MongoDB
    try:
        # Nota: Si usas Docker, cambia 'localhost' por el nombre del contenedor
        client = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=3000)
        db = client["Canasta_db"]
        coleccion = db["Retail_A"]
        coleccion.insert_many(datos_finales)
        print(f"📦 {len(datos_finales)} registros guardados en MongoDB.")
    except Exception as e:
        print(f"⚠️ Alerta MongoDB: No se pudo conectar (localhost:27017). Error: {e}")

    # Backup JSON (Siempre se ejecuta)
    try:
        # IMPORTANTE: Limpiamos los datos para que sean compatibles con JSON
        datos_limpios = serializar_datos(datos_finales)
        archivo_nombre = f"tottus_{CATEGORIA.lower()}.json"
        with open(archivo_nombre, "w", encoding="utf-8") as f:
            json.dump(datos_limpios, f, ensure_ascii=False, indent=2)
        print(f"💾 Backup JSON creado: {archivo_nombre}")
    except Exception as e:
        print(f"❌ Error al crear el archivo JSON: {e}")
else:
    print("❌ No se pudo extraer ninguna información. Verifica la conexión o el sitio web.")

✅ Entorno limpio.
🚀 Cargando Carnes en Tottus...
🖱️ Bajando para forzar el renderizado...
🖱️ Bajando por la página para activar la grilla...
   Bajando... 1/8
   Bajando... 2/8
   Bajando... 3/8
   Bajando... 4/8
   Bajando... 5/8
   Bajando... 6/8
   Bajando... 7/8
   Bajando... 8/8
🔍 Iniciando escaneo profundo de texto...
📦 Bloques detectados en escaneo profundo: 27
🥩 [1] TOTTUS NACIONAL | $ 10.232
🥩 [2] TOTTUS BLACK | $ 15.192
🥩 [3] ESTANCIA 92 Teclas de Lomo Importado Cat | $ 9.990
🥩 [4] ESTANCIA 92 Palanca Importado Cat V al V | $ 10.990
🥩 [5] ESTANCIA 92 Sobrecostilla (Denver Steak) | $ 10.490
🥩 [6] Envío rápido | $ 14.792
🥩 [7] ESTANCIA 92 Poncho Parrillero Importado  | $ 9.990
🥩 [8] Envío rápido | $ 10.392
🥩 [9] Fósforos de Seguridad | $ 1.590
🥩 [10] Por TOTTUS Envío rápido | $ 1.820
🥩 [11] Iniciador Fuego Blanco | $ 2.690
🥩 [12] LA GRAN ESTANCIA | $ 13.832
🥩 [13] LA GRAN ESTANCIA | $ 11.592
🥩 [14] Envío rápido | $ 5.990
🥩 [15] TOTTUS IMPORTADO | $ 8.792
🥩 [16] Envío rápido | $

NameError: name 'productos' is not defined

In [12]:
# --- SCRAPING TOTTUS FRUTAS Y VERDURAS ---
import os
import time
import random
import re
import json
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bson import ObjectId

# --- LIMPIEZA DE ENTORNO ---
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("✅ Entorno limpio.")

# --- VARIABLES ---
NOMBRE_GRUPO = "Vannessa"
CATEGORIA = "Frutas y Verduras"
URL_TOTTUS = "https://www.tottus.cl/tottus-cl/lista/CATG27070/Frutas-y-Verduras"
datos_finales = []

# --- CONFIGURACIÓN ---
options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")

# Anti-bloqueo básico
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

# User-Agent realista
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
)

def serializar_datos(lista):
    for item in lista:
        if "_id" in item:
            item["_id"] = str(item["_id"])
    return lista

driver = webdriver.Chrome(options=options)

try:
    print(f"🚀 Cargando {CATEGORIA} en Tottus...")
    driver.get(URL_TOTTUS)

    # ⛔ IMPORTANTE: dar tiempo a que cargue ubicación + contenido dinámico
    print("⏳ Esperando carga inicial...")
    time.sleep(8)

    # --- SCROLL HUMANO ---
    print("🖱️ Scroll humano...")
    for i in range(12):
        driver.execute_script(f"window.scrollBy(0, {random.randint(400,800)});")
        print(f"   Scroll {i+1}")
        time.sleep(random.uniform(1.5, 3))

    # --- SCROLL FINAL FUERTE ---
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5)

    # --- EXTRACCIÓN PROFUNDA ---
    print("🔍 Escaneando productos...")

    script_total = """
    let listado = [];
    let todos = document.querySelectorAll('span, div, b, p, strong');

    todos.forEach(el => {
        if (el.innerText.includes('$') && el.innerText.length < 25) {
            let contenedor = el.parentElement;

            for (let i = 0; i < 6; i++) {
                if (contenedor && contenedor.innerText.length > 60) {
                    listado.push(contenedor.innerText);
                    break;
                }
                contenedor = contenedor.parentElement;
            }
        }
    });

    return [...new Set(listado)];
    """

    bloques_texto = driver.execute_script(script_total)
    print(f"📦 Bloques encontrados: {len(bloques_texto)}")

    # --- PROCESAMIENTO ---
    for i, texto_raw in enumerate(bloques_texto):
        try:
            lineas = [l.strip() for l in texto_raw.split('\n') if len(l.strip()) > 1]

            # Precio
            precio = next((l for l in lineas if '$' in l), None)

            # FILTRO MEJORADO (CLAVE)
            palabras_basura = [
                "DESPACHO", "RETIRO", "AGREGAR", "VER MÁS", "ORDENAR",
                "FILTRAR", "CLIENTE", "ENVÍO", "PRECIOS", "INCREÍBLES"
            ]

            candidatos = [
                l for l in lineas
                if '$' not in l
                and len(l) > 10
                and not any(p in l.upper() for p in palabras_basura)
            ]

            if precio and candidatos:
                nombre_final = candidatos[0]

                if len(nombre_final) < 15 and len(candidatos) > 1:
                    nombre_final = f"{nombre_final} {candidatos[1]}"

                valor_limpio = re.sub(r'[^\d]', '', precio)

                datos_finales.append({
                    "identificador": nombre_final,
                    "valor": precio,
                    "valor_limpio": float(valor_limpio),
                    "categoria": CATEGORIA,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO,
                    "tienda": "Tottus",
                    "url": URL_TOTTUS
                })

                print(f"🥬 [{len(datos_finales)}] {nombre_final[:50]} | {precio}")

        except:
            continue

finally:
    driver.quit()
    print("\n🛑 Navegador cerrado.")

# --- GUARDADO ---
if datos_finales:
    try:
        client = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=3000)
        db = client["Canasta_db"]
        coleccion = db["Retail_A"]
        coleccion.insert_many(datos_finales)
        print(f"📦 {len(datos_finales)} registros guardados en MongoDB.")
    except Exception as e:
        print(f"⚠️ MongoDB no disponible: {e}")

    try:
        datos_limpios = serializar_datos(datos_finales)
        archivo_nombre = "tottus_frutas_verduras.json"

        with open(archivo_nombre, "w", encoding="utf-8") as f:
            json.dump(datos_limpios, f, ensure_ascii=False, indent=2)

        print(f"💾 Backup creado: {archivo_nombre}")

    except Exception as e:
        print(f"❌ Error JSON: {e}")

else:
    print("❌ No se extrajeron productos.")

✅ Entorno limpio.
🚀 Cargando Frutas y Verduras en Tottus...
⏳ Esperando carga inicial...
🖱️ Scroll humano...
   Scroll 1
   Scroll 2
   Scroll 3
   Scroll 4
   Scroll 5
   Scroll 6
   Scroll 7
   Scroll 8
   Scroll 9
   Scroll 10
   Scroll 11
   Scroll 12
🔍 Escaneando productos...
📦 Bloques encontrados: 46
🥬 [1] FUNDO SOFRUCO Ciruelas Deshidratadas sin Carozo 30 | $ 2.790
🥬 [2] Tomate Cocktail Regy Envasado 250 g | $ 1.990
🥬 [3] Maracuyá a Granel | $ 4.950
🥬 [4] FUNDO SOFRUCO Ciruelas Deshidratadas sin Carozo 50 | $ 4.990
🥬 [5] Cebolla Tubular en Malla 4 Un | $ 990
🥬 [6] Mini Pimientos Dulces en Bolsa 200 g | $ 1.990
🥬 [7] Papa Souffle en Malla 2 Kg | $ 2.490
🥬 [8] Rúcula Orgánica en Bolsa 200 g | $ 2.550
🥬 [9] Pimentón Naranjo Hidropónico | $ 1.550
🥬 [10] Cebolla Tubular en Malla 3 Un | $ 1.720
🥬 [11] Ajo Desgranado en Malla 150 g | $ 1.420
🥬 [12] Tomate Beef a Granel | $ 2.490
🥬 [13] Tomate Triple Tasty Envasado 500 g | $ 4.250
🥬 [14] FUNDO SOFRUCO Ciruelas Deshidratadas sin Carozo 3

In [13]:
# --- SCRAPING TOTTUS DESPENSA ---
import os
import time
import random
import re
import json
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bson import ObjectId

# --- LIMPIEZA ---
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("✅ Entorno limpio.")

# --- VARIABLES ---
NOMBRE_GRUPO = "Vannessa"
CATEGORIA = "Despensa"
URL_TOTTUS = "https://www.tottus.cl/tottus-cl/lista/CATG27055/Despensa"
datos_finales = []

# --- CONFIG ---
options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
)

def serializar_datos(lista):
    for item in lista:
        if "_id" in item:
            item["_id"] = str(item["_id"])
    return lista

driver = webdriver.Chrome(options=options)

try:
    print(f"🚀 Cargando {CATEGORIA} en Tottus...")
    driver.get(URL_TOTTUS)

    # ⛔ IMPORTANTE: tiempo para ubicación + carga dinámica
    print("⏳ Esperando carga inicial...")
    time.sleep(8)

    # --- SCROLL HUMANO ---
    print("🖱️ Scroll humano...")
    for i in range(10):
        driver.execute_script(f"window.scrollBy(0, {random.randint(500,900)});")
        print(f"   Scroll {i+1}")
        time.sleep(random.uniform(1.5, 2.5))

    # Scroll final
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(4)

    # --- EXTRACCIÓN ---
    print("🔍 Escaneando productos...")

    script_total = """
    let listado = [];
    let todos = document.querySelectorAll('span, div, b, p, strong');

    todos.forEach(el => {
        if (el.innerText.includes('$') && el.innerText.length < 25) {
            let contenedor = el.parentElement;

            for (let i = 0; i < 6; i++) {
                if (contenedor && contenedor.innerText.length > 60) {
                    listado.push(contenedor.innerText);
                    break;
                }
                contenedor = contenedor.parentElement;
            }
        }
    });

    return [...new Set(listado)];
    """

    bloques_texto = driver.execute_script(script_total)
    print(f"📦 Bloques encontrados: {len(bloques_texto)}")

    # --- PROCESAMIENTO ---
    for texto_raw in bloques_texto:
        try:
            lineas = [l.strip() for l in texto_raw.split('\n') if len(l.strip()) > 1]

            precio = next((l for l in lineas if '$' in l), None)

            # 🔥 FILTRO DESPENSA (menos agresivo que lácteos)
            palabras_basura = [
                "DESPACHO", "RETIRO", "AGREGAR", "VER MÁS",
                "ORDENAR", "FILTRAR", "CLIENTE", "ENVÍO"
            ]

            candidatos = [
                l for l in lineas
                if '$' not in l
                and len(l) > 8
                and not any(p in l.upper() for p in palabras_basura)
            ]

            if precio and candidatos:
                nombre_final = candidatos[0]

                if len(nombre_final) < 15 and len(candidatos) > 1:
                    nombre_final = f"{nombre_final} {candidatos[1]}"

                valor_limpio = re.sub(r'[^\d]', '', precio)

                datos_finales.append({
                    "identificador": nombre_final,
                    "valor": precio,
                    "valor_limpio": float(valor_limpio),
                    "categoria": CATEGORIA,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO,
                    "tienda": "Tottus",
                    "url": URL_TOTTUS
                })

                print(f"🛒 [{len(datos_finales)}] {nombre_final[:50]} | {precio}")

        except:
            continue

finally:
    driver.quit()
    print("\n🛑 Navegador cerrado.")

# --- GUARDADO ---
if datos_finales:
    try:
        client = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=3000)
        db = client["Canasta_db"]
        coleccion = db["Retail_A"]
        coleccion.insert_many(datos_finales)
        print(f"📦 {len(datos_finales)} guardados en MongoDB.")
    except Exception as e:
        print(f"⚠️ MongoDB error: {e}")

    try:
        archivo = "tottus_despensa.json"
        with open(archivo, "w", encoding="utf-8") as f:
            json.dump(serializar_datos(datos_finales), f, ensure_ascii=False, indent=2)
        print(f"💾 Backup creado: {archivo}")
    except Exception as e:
        print(f"❌ Error JSON: {e}")

else:
    print("❌ No se extrajeron productos.")

✅ Entorno limpio.
🚀 Cargando Despensa en Tottus...
⏳ Esperando carga inicial...
🖱️ Scroll humano...
   Scroll 1
   Scroll 2
   Scroll 3
   Scroll 4
   Scroll 5
   Scroll 6
   Scroll 7
   Scroll 8
   Scroll 9
   Scroll 10
🔍 Escaneando productos...
📦 Bloques encontrados: 57
🛒 [1] Jurel al Natural San Jose 425 g | $ 2.250
🛒 [2] Atún Lomitos al Natural San Jose 140 g | $ 1.590
🛒 [3] Gelatina Daily con Stevia Sabor Frambuesa 22.5 g | $ 770
🛒 [4] EXTRACTO PAN DE PASCUA 60 CC | $ 1.890
🛒 [5] Pasta Fusilli Armando 500 g | $ 1.290
🛒 [6] Pasta Penne Armando 500 g | $ 1.290
🛒 [7] Lomo Jurel al Agua San Jose 160 g | $ 1.450
🛒 [8] Atún Lomitos en Aceite San Jose 140 g | $ 1.590
🛒 [9] DOS CABALLOS Puré de Manzana Dos Caballos 600 g | $ 2.050
🛒 [10] Pasta Spaghetti Armando 500 g | $ 1.290
🛒 [11] CLUB SOCIAL Pack Galleta Club Social Original 12 x | $ 2.290
🛒 [12] EVERCRISP Maní Maniax Original | $ 1.990
🛒 [13] Papas Fritas Sal De Mar Wavy | $ 2.250
🛒 [14] Por TOTTUS | $ 3.050
🛒 [15] Estragón Deshidrat